Found An Album called Asexual Anger by Love Sex Machine. Not all of the songs have lyrics written down for it unfortunately, but the ones that do all seem to have some theme relating to being asexual. 

In [ ]:
import requests
import time
import os
import re  # this is for regex substitutions! 
from dotenv import load_dotenv, find_dotenv
from bs4 import BeautifulSoup

load_dotenv(find_dotenv())

In [ ]:
# ebb: Here are two scraping functions that work as of 30 March 2026! 
# READ FROM THE BOTTOM TO TOP. 
# Supply your album title and define your relative path to your output directory where you want to store lyrics text files (one text file for each song).
# That information is needed to start up the get_lyrics_from_album_url(album_url, output_dir)
# And as that function pulls song titles from the album page, it then triggers the get_lyrics() function, sending it the song URL.
# get_lyrics(), the first function you see here, delivers the song lyrics back to the main function, which then writes the lyrics to output.
# Genius keeps changing its HTML page metadata, so we probably will have to keep updating this cell over time.
def get_lyrics(song_url):
    r = requests.get(song_url)
    soup = BeautifulSoup(r.text, "html.parser")
    
    lyric_divs = soup.find_all("div", attrs={"data-lyrics-container": "true"})
    
    lyrics = []
    for div in lyric_divs:
        for br in div.find_all("br"):
            br.replace_with("\n")
        text = div.get_text()
        
        if not text.strip():
            continue
        
        # If the div contains a bracket marker, trim everything before the first one
        first_bracket = text.find("[")
        if first_bracket != -1:
            text = text[first_bracket:]
            
        # ebb: The next two lines remove the square-bracketed stuff 
        # like `[Verse 1]`, `[Outro]`, etc. with the regular expression`\[.*?\]` 
        # If you want to KEEP these for use in tagging later for an XML project, 
        # comment out the next two lines. 
        text = re.sub(r'\[.*?\]', '', text)
        text = re.sub(r'\n{3,}', '\n\n', text).strip()
        lyrics.append(text)
    
    return "\n".join(lyrics).strip()
    
def get_lyrics_from_album_url(album_url, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    
    
    r = requests.get(album_url)
    soup = BeautifulSoup(r.text, "html.parser")
    
    # Only look for lyrics links inside chart_row-content containers
    track_rows = soup.find_all("div", class_="chart_row-content")
    
    corpus = {}
    for row in track_rows:
        link = row.find("a", href=lambda v: v and v.endswith("-lyrics"))
        if link:
            song_url = link["href"]
            song_title = link.get_text(strip=True)
            print(f"Fetching: {song_title}")
            lyrics = get_lyrics(song_url)
            # This runs the function above!!!
            corpus[song_title] = lyrics
            
            # Clean up (remove spaces, path separators, etc. from the song title) for use as a filename
            # Also remove the "Lyrics" from the end of the song title with a regex subsitutitons on Lyrics$ (at the end of the string), 
            # and removing punctuation from titles with a regex character class.
            safe_title = song_title.replace("/", "-").replace(" ", "_").strip()
            safe_title = re.sub(r'Lyrics$', '', safe_title).strip()
            safe_title = re.sub(r"[,.'!?;:]", '', safe_title)
            filepath = os.path.join(output_dir, f"{safe_title}.txt")
            
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(lyrics)
                
            print(f"  Saved: {filepath}")
            time.sleep(1)
    
    return corpus

# Usage
album_url = "https://genius.com/Love-sex-machine-asexual-anger-lyrics"
output_dir = "/lyricOutput/asexual-anger"

corpus = get_lyrics_from_album_url(album_url, output_dir)